# STEP 1: Install Dependencies (with AgentOps)

In [ ]:
!pip install -q crewai crewai-tools agentops duckduckgo_search yfinance pandas beautifulsoup4 python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.8/820.8 kB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.6/309.6 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# STEP 2: Imports (add AgentOps)

In [ ]:
import os
from dotenv import load_dotenv
from getpass import getpass
import yfinance as yf
from duckduckgo_search import DDGS
from crewai import Agent, Task, Crew, LLM
from crewai.tools import BaseTool
import agentops
from google.colab import userdata

# Load environment variables
load_dotenv()


False

# STEP 3: Load Keys (Gemini + AgentOps)

In [ ]:
# Load Gemini API Key
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Load AgentOps API Key
os.environ["AGENTOPS_API_KEY"] = userdata.get("AGENTOPS_API_KEY")

# Initialize AgentOps
agentops.init(api_key=os.environ["AGENTOPS_API_KEY"])


#  STEP 4: LLM Setup (Gemini via CrewAI)

In [ ]:
llm = LLM(
    model="gpt-4o-mini",  # Ensure this model is valid and accessible
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0.7
)


# STEP 5: Define Tools

In [ ]:
class StockSearchTool(BaseTool):
    name: str = "StockNewsSearcher"
    description: str = "Search for the latest news and updates about a stock using DuckDuckGo"

    def _run(self, query: str) -> str:
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=2)
            return "\n".join([r['body'] for r in results])

class YahooFinanceTool(BaseTool):
    name: str = "YahooFinanceFetcher"
    description: str = "Get the latest 1-month stock price history for a given ticker using yFinance"

    def _run(self, ticker: str) -> str:
        stock = yf.Ticker(ticker)
        hist = stock.history(period="1mo")
        return hist.tail(3).to_string()


# STEP 6: Define Agents (Monitored by AgentOps)

In [ ]:
stock_analyst = Agent(
    role='Stock Analyst',
    goal='Analyze recent stock data and news',
    backstory='Expert in financial trends, macro indicators, and company performance',
    verbose=True,
    allow_delegation=False,
    llm=llm,
    agentops_enabled=True  # ✅ Enables AgentOps logging
)

report_writer = Agent(
    role='Report Generator',
    goal='Write investor-friendly summaries of stock analysis',
    backstory='Professional writer with expertise in finance reporting',
    verbose=True,
    allow_delegation=False,
    llm=llm,
    agentops_enabled=True  # ✅ Enables AgentOps logging
)


# STEP 7: Define Tasks

In [ ]:
search_tool = StockSearchTool()
finance_tool = YahooFinanceTool()

search_task = Task(
    description="Search latest news and updates about the stock 'AAPL' using DuckDuckGo.",
    expected_output="Summarized news highlights for Apple stock.",
    agent=stock_analyst,
    tools=[search_tool]
)

analysis_task = Task(
    description="Analyze Apple stock price trends using yFinance.",
    expected_output="Key trends and technical highlights for the past month.",
    agent=stock_analyst,
    tools=[finance_tool]
)

report_task = Task(
    description="Write a clean investor report using previous analysis and news insights.",
    expected_output="Concise report with market summary and investment outlook.",
    agent=report_writer
)


# STEP 8: Assemble the Crew

In [ ]:
crew = Crew(
    agents=[stock_analyst, report_writer],
    tasks=[search_task, analysis_task, report_task],
    verbose=False  # Keep output clean
)


# STEP 9: Run the Agent Crew with AgentOps Logging

In [ ]:
result = await crew.kickoff_async()

print("\n📊 Final Stock Analysis Report:\n")
print(result)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst                                                                                           │
│                                                                                                                 │
│  Task: Search latest news and updates about the stock 'AAPL' using DuckDuckGo.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/tmp/ipykernel_1615/4232134819.py:6: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Tool stock_news_searcher executed with result: Apple Inc. (NASDAQ: AAPL) shares decreased sharply as investors focused on weaker forward guidance and rising costs, despite the …
Get the latest Apple Inc. (AAPL) stock news and headlines to help you...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

[Finalize] todos_count=0, todos_with_results=0


...


📊 Final Stock Analysis Report:

### Investor Report: Apple Inc. (AAPL) Stock Analysis

#### Market Summary:
Apple Inc. (NASDAQ: AAPL) has faced considerable downward pressure over the past month, with shares declining sharply due to concerns regarding weaker forward guidance and rising operational costs. The stock's performance has been characterized by significant volatility and heightened trading volumes, indicating a cautious sentiment among investors.

#### Key Trends:
1. **Overall Performance**:
   - Apple shares have seen a notable decline, reflecting investor anxiety surrounding future earnings expectations. This has been further exacerbated by rising costs impacting operational margins.

2. **Price Movement**:
   - On **July 29, 2026**, AAPL opened at **$339.73**, reached a high of **$344.57**, and closed at **$338.19**.
   - The following day, **July 30, 2026**, the stock opened lower at **$333.10**, peaked at **$334.75**, and closed at **$333.43**.
   - By **July 31, 2026**,

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x78e27b8133f0>
/tmp/ipykernel_1615/3526949887.py:1: RuntimeWarning: coroutine 'Crew.kickoff_async' was never awaited
  result = await crew.kickoff_async()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and s

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag